# Bước 03/06 — Tạo V6.5 external stage → `v6p5_ext_<12hex>`

Open Images V7 TRAIN (Balloon/Kite/Plastic bag/Bird/Person, detections-only + GrabCut, 250/class) · MPCD (broken → broken_strand) · Power Equipment (dx_dg/yw/nw, dx_sg chỉ review) · synthetic foreign_object 100/subtype trên TRAIN background · exact-SHA + pHash QA · review manifest · đóng gói + freeze + upload. Checkpoint theo class / subtype.

| | Camber Stash |
|---|---|
| **Input** | `Open Images V7 (internet)` |
| **Input** | `MPCD Google Drive 1KyciMmwL2_p_-mwckHFqG5fkBA1jzATv` |
| **Input** | `HF sxiong/Power-equipment-image-dataset` |
| **Input** | `kết quả bước 02` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/v6p5_ext_<12hex>/v6p5_ext_<12hex>.freeze.json` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/v6p5_ext_<12hex>/v6p5_ext_<12hex>_overlay.tar.gz` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/v6p5_ext_<12hex>/v6p5_ext_<12hex>_qa.zip` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/mpcd/mpcd_broken_strand_stage.tar.gz` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/power_equipment/power_equipment_stage.tar.gz` |

Chạy lại notebook bất cứ lúc nào: bước đã xong + đúng config sẽ được **reuse**, bước dở dang sẽ **resume**.
Bước tiếp theo: **04_review_and_compose_v6p5**

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
# ===== DATASET TRÊN CAMBER — điền tag nếu muốn chọn tay, None = tự động =====
PARENT_DATASET = "v6p2e_23562a45b343"   # V6.2E: cố định, KHÔNG đổi
EXTERNAL_STAGE = None   # vd "v6p5_ext_0123456789ab"     (tạo ở bước 03)
COMPOSE        = None   # vd "v6p5_compose_0123456789ab" (tạo ở bước 04)
REQUIRE_CAMBER = True   # lưu chính trên Camber -> dừng nếu chưa đăng nhập

from yolov11sdi.camber import CamberClient
from yolov11sdi.environment import load_camber_api_key
from yolov11sdi.reporting import status
cfg = pipeline.load(stage_tag=EXTERNAL_STAGE, compose_tag=COMPOSE, parent_tag=PARENT_DATASET)
load_camber_api_key(paths.env)
cam = CamberClient(paths, cfg.get("camber.stash_prefix"), cfg.runtime("camber_bin"))
if REQUIRE_CAMBER and not cam.available():
    raise RuntimeError("Camber chưa sẵn sàng: chạy `camber login` hoặc export CAMBER_API_KEY "
                       "(CAMBER_BIN=... nếu CLI không nằm trong PATH). Chỉ chạy local: REQUIRE_CAMBER=False")
display(status.dataset_card(cfg, paths, cam))

In [ ]:
for s in ["parent", "openimages", "mpcd", "power_equipment", "synthetic", "dedup"]:
    print(pipeline.run_stage(s, cfg, force=FORCE_REBUILD, dry_run=DRY_RUN))
d = store.load("dedup").summary if store.load("dedup") else {}
print("exact SHA trùng V6.2E (HARD gate):", d.get("exact_duplicates"))
print("pHash gần trùng VAL/TEST (cần review):", d.get("phash_candidates"))

In [ ]:
outcome = pipeline.run_stage("external_freeze", cfg, force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
# Kiểm tra output đã thật sự nằm trên Camber chưa
display(pipeline.verify_on_camber("external_freeze", cfg))

In [ ]:
contacts = sorted((paths.qa / "v6p5_external" / "contacts").glob("*.jpg"))
print(len(contacts), "contact sheets")
for p in contacts[:12]:
    print(p.name); display(IPImage(filename=str(p), width=640))

Ghi lại tag `v6p5_ext_...` ở bảng trên — bước 04 tự lấy, hoặc điền vào `EXTERNAL_STAGE`.